# eVED (Ann Arbor) → TTE format

**Source:** `git clone https://Datarepo@bitbucket.org/datarepo/eved_dataset.git`
— the GitHub link printed in the paper (arXiv 2203.08630) is dead, the Bitbucket
clone is alive. **Verified: it clones (1.3 GB) and `data/eVED.zip` holds 54 weekly
CSVs, 5.8 GB uncompressed.** Extends VED (`github.com/gsoh/VED`, Apache-2.0);
**the eVED licence itself is not stated** — check before redistributing anything.

383 private cars in Ann Arbor, Nov 2017 – Nov 2018, ~22 M records.

**Why it is the best of the new candidates:** sampling is **~1 Hz**
(median 0.6–0.9 s between fixes — measured on the real files), trips are explicit
(`VehId`, `Trip`), and every record already carries a **calibrated on-road
coordinate** (`Matchted Latitude[deg]` / `Matched Longitude[deg]`, non-null for
100 % of the rows we checked), which makes map matching almost free.

**What the appendix gets slightly wrong:** eVED does *not* hand you an edge id
sequence. It hands you snapped coordinates plus road attributes (speed limit,
elevation, gradient, intersection / bus stop / crossing flags). The OSM segment
ids still have to be produced here — but starting from on-road coordinates, so
the matcher has an easy job.

**Time base (worked out from the data, not documented):** `DayNum` is *constant
within a trip* — it is the trip start, in days since **2017-11-01**, 1-based.
`Timestamp(ms)` restarts at 0 for every trip. So
`epoch = epoch(2017-11-01) + (DayNum - 1) * 86400 + Timestamp(ms) / 1000`.

## Target format (the "gold" contract)

Taken from the Harbin files in `datasets.zip`, with the Omsk file naming:

| file | contents |
|---|---|
| `matched_trips_<city>.csv` | unnamed index, `Id`, `Coordinates`, `OSMids`, `Timestamps`, `Total_time` |
| `edge_list_directed_<city>.csv` | `osmid_u`, `osmid_v` — directed transitions between road segments |
| `road_network_unique_osmids_<city>.geojson` | one `LineString` per segment, `osmid` / `original_osmid` / `is_duplicate` / `duplicate_index` / `length` / `highway` / ... |

`Coordinates`, `OSMids` and `Timestamps` are Python-literal lists of **equal
length — one entry per GPS fix**: `(lon, lat)` floats, the segment id the fix was
matched to (a string), and the unix timestamp in seconds.
`Total_time = Timestamps[-1] - Timestamps[0]`, in seconds.

In [ ]:
CITY    = "ann_arbor"
RAW_DIR = "eved_dataset/data/x/eVED"   # unzip data/eVED.zip here
OUT     = "."
GRAPHML = "ann_arbor_drive.graphml"

BBOX = (-83.85, 42.20, -83.62, 42.35)      # (left, bottom, right, top)
EPOCH0 = "2017-11-01"                      # DayNum == 1

RESAMPLE_S  = 1      # keep at most one fix per this many seconds (1 = keep 1 Hz)
MAX_TRIPS   = None   # trips are only ~30 k in total, no need to sample
RANDOM_SEED = 0

MIN_POINTS       = 8
MIN_SECONDS      = 120
MAX_SECONDS      = 5400
MIN_METERS       = 800
MAX_SPEED_MS     = 45.0
MEAN_SPEED_RANGE = (1.0, 25.0)

MAX_GAP_S        = 300
STILL_M          = 25
STOP_S           = 180

SNAP_RADIUS_M    = 60
SNAP_SIGMA_M     = 20
MAX_SNAP_M       = 50
MIN_MATCHED_FRAC = 0.8
MIN_CONNECTIVITY = 0.99       # after route filling this should be exactly 1.0
BETA_M           = 10.0       # transition scale: |network dist - straight dist| / beta
MAX_ROUTE_FACTOR = 4.0        # reject a detour longer than this x the straight gap
MAX_MEDIAN_STEP_M = None      # metres between fixes; None = keep everything.
                              # At a fixed cadence this is a SPEED filter — see below.


In [ ]:
# pip install pandas numpy osmnx geopandas shapely networkx tqdm
import ast, json, math, os, glob
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import Point, mapping
from pyproj import Transformer
from tqdm.auto import tqdm

## Helpers

In [ ]:
# --- unique OSM ids -----------------------------------------------------------
def assign_unique_osmids(G):
    """(u, v, key) -> {unique_osmid, original_osmid, is_duplicate, duplicate_index}.

    One OSM way is split into many graph edges, so `osmid` is not unique.  The
    gold files use `<way_id>` when a way appears once and `<way_id>_<i>`
    (i = 1, 2, ...) for every edge of a way that appears several times -- see
    `Harbin_edge_list.csv` ('858780553' next to '705148575_1').
    """
    base = {}
    for u, v, k, d in G.edges(keys=True, data=True):
        o = d.get("osmid")
        if isinstance(o, (list, tuple, set)):
            o = sorted(o)[0]
        base[(u, v, k)] = str(o)

    counts = pd.Series(list(base.values())).value_counts().to_dict()
    seen, out = {}, {}
    for e, b in base.items():
        if counts[b] == 1:
            out[e] = dict(unique_osmid=b, original_osmid=b,
                          is_duplicate=False, duplicate_index=0)
        else:
            i = seen.get(b, 0) + 1
            seen[b] = i
            out[e] = dict(unique_osmid=f"{b}_{i}", original_osmid=b,
                          is_duplicate=True, duplicate_index=i)
    return out


# --- edge_list_directed_<city>.csv --------------------------------------------
def build_edge_list_directed(G, uid):
    """Directed transitions between segments that share a node (Harbin style)."""
    inc, out = {}, {}
    for u, v, k in G.edges(keys=True):
        out.setdefault(u, []).append((u, v, k))
        inc.setdefault(v, []).append((u, v, k))
    rows = set()
    for node in set(inc) & set(out):
        for e1 in inc[node]:
            for e2 in out[node]:
                if e1 == e2:
                    continue
                a, b = uid[e1]["unique_osmid"], uid[e2]["unique_osmid"]
                if a != b:
                    rows.add((a, b))
    return pd.DataFrame(sorted(rows), columns=["osmid_u", "osmid_v"])


# --- road_network_unique_osmids_<city>.geojson --------------------------------
_KEEP = ["bridge", "highway", "lanes", "name", "oneway", "reversed",
         "junction", "ref", "tunnel", "maxspeed", "access", "width"]


def _clean(val):
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    if isinstance(val, (list, tuple)):
        val = [str(x) for x in val]
        return val or None
    return val if isinstance(val, bool) else str(val)


def build_road_network_geojson(G, uid, path):
    """One LineString per graph edge with the Harbin property set."""
    Gu = G if str(G.graph.get("crs", "")).lower() in ("epsg:4326", "wgs84") \
        else ox.projection.project_graph(G, to_latlong=True)
    edges = ox.convert.graph_to_gdfs(Gu, nodes=False, edges=True, fill_edge_geometry=True)
    feats = []
    for (u, v, k), row in edges.iterrows():
        info = uid[(u, v, k)]
        props = {"u": int(u), "v": int(v), "key": int(k),
                 "osmid": info["original_osmid"],
                 "unique_osmid": info["unique_osmid"]}
        for c in _KEEP:
            props[c] = _clean(row[c]) if c in edges.columns else None
        props["length"] = float(row["length"])
        props["original_osmid"] = info["original_osmid"]
        props["is_duplicate"] = info["is_duplicate"]
        props["duplicate_index"] = info["duplicate_index"]
        props = {a: b for a, b in props.items() if b is not None}
        feats.append({"type": "Feature", "properties": props,
                      "geometry": mapping(row["geometry"])})
    fc = {"type": "FeatureCollection", "name": "road_network_unique_osmids",
          "crs": {"type": "name",
                  "properties": {"name": "urn:ogc:def:crs:OGC:1.3:CRS84"}},
          "features": feats}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(fc, f, ensure_ascii=False)
    return len(feats)


# --- map matching -------------------------------------------------------------
class Matcher:
    """HMM map matcher that returns a *connected route*, not one edge per fix.

    emission   = snap distance, Gaussian with `sigma_m`
    transition = |network distance - straight-line distance| / `beta_m`
                 (Newson & Krumm)

    The old hop-count transition (same / 1-hop / 2-hop) silently assumed the
    vehicle never travelled more than two edges between two fixes.  That holds
    at 1-7 s sampling and breaks completely at 30-60 s, where a car crosses
    half a dozen blocks between fixes.  A distance-based transition has no such
    assumption.

    `match_route` also inserts the edges the vehicle must have crossed between
    two fixes, so consecutive segments in the output are genuinely adjacent and
    `connectivity()` measures something real instead of the sampling rate.
    """

    def __init__(self, G, radius_m=60.0, sigma_m=20.0, k_candidates=8,
                 beta_m=10.0, max_route_factor=4.0, min_route_slack_m=300.0):
        projected = str(G.graph.get("crs", "")).lower() not in ("epsg:4326", "wgs84", "")
        self.Gp = G if projected else ox.projection.project_graph(G)
        self.crs = self.Gp.graph["crs"]
        self.edges = ox.convert.graph_to_gdfs(self.Gp, nodes=False, edges=True,
                                              fill_edge_geometry=True)
        self.index = list(self.edges.index)
        self.sindex = self.edges.sindex
        self.geom = dict(zip(self.index, self.edges.geometry))
        self.length = {e: float(l) for e, l in zip(self.index, self.edges["length"])}
        self.radius, self.sigma, self.k = radius_m, sigma_m, k_candidates
        self.beta = beta_m
        self.max_factor, self.slack = max_route_factor, min_route_slack_m
        self.succ = {}
        for u, v, k in self.Gp.edges(keys=True):
            self.succ.setdefault(u, []).append((u, v, k))
        self._to_lonlat = Transformer.from_crs(self.crs, "EPSG:4326",
                                               always_xy=True).transform
        self._dij = {}

    # ---------------------------------------------------------------- helpers
    def _dijkstra(self, source, cutoff):
        """Cached single-source distances; recomputed only if the cutoff grows."""
        hit = self._dij.get(source)
        if hit is None or hit[0] < cutoff:
            dist = nx.single_source_dijkstra_path_length(
                self.Gp, source, cutoff=cutoff, weight="length")
            self._dij[source] = (cutoff, dist)
            return dist
        return hit[1]

    def _net_dist(self, e1, e2, cutoff):
        if e1 == e2 or e1[1] == e2[0]:
            return 0.0
        return self._dijkstra(e1[1], cutoff).get(e2[0], np.inf)

    def _candidates(self, xs, ys):
        out = []
        for x, y in zip(xs, ys):
            p = Point(x, y)
            hits = self.sindex.query(p.buffer(self.radius), predicate="intersects")
            if len(hits) == 0:
                out.append([])
                continue
            cand = [(self.index[i], self.geom[self.index[i]].distance(p))
                    for i in np.atleast_1d(hits)]
            cand.sort(key=lambda t: t[1])
            out.append(cand[: self.k])
        return out

    # ---------------------------------------------------------------- viterbi
    def match(self, lon, lat):
        """Best edge per fix.  Returns (edges, snap_dist, keep_mask).

        A fix with no edge within `radius_m` is dropped from `keep_mask` rather
        than failing the whole trip.
        """
        pts = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs(self.crs)
        X, Y = pts.x.values, pts.y.values
        cands = self._candidates(X, Y)

        keep = np.array([len(c) > 0 for c in cands])
        if keep.sum() < 2:
            return None, None, keep
        idx = np.flatnonzero(keep)
        cands = [cands[i] for i in idx]
        X, Y = X[idx], Y[idx]

        score = {e: 0.5 * (d / self.sigma) ** 2 for e, d in cands[0]}
        back = [{}]
        for t in range(1, len(cands)):
            gc = float(np.hypot(X[t] - X[t - 1], Y[t] - Y[t - 1]))
            cutoff = max(gc * self.max_factor, self.slack)
            new, bp = {}, {}
            for e2, d2 in cands[t]:
                best_e, best_s = None, np.inf
                for e1, s1 in score.items():
                    nd = self._net_dist(e1, e2, cutoff)
                    s = s1 + (abs(nd - gc) / self.beta if np.isfinite(nd) else 1e6)
                    if s < best_s:
                        best_s, best_e = s, e1
                new[e2] = best_s + 0.5 * (d2 / self.sigma) ** 2
                bp[e2] = best_e
            score = new
            back.append(bp)

        e = min(score, key=score.get)
        path = [e]
        for t in range(len(cands) - 1, 0, -1):
            e = back[t][e]
            path.append(e)
        path.reverse()
        dist = np.array([dict(c).get(e, np.nan) for c, e in zip(cands, path)])
        return path, dist, keep

    # ------------------------------------------------------------ route fill
    def fill_route(self, path, gc_gaps):
        """Edges strictly between consecutive fixes.  Returns (inserted, ok).

        A detour far longer than the straight-line gap means the two fixes were
        not really matched to the same journey -> the trip is rejected instead
        of being stitched together with an invented loop.
        """
        inserted, ok = [], True
        for (e1, e2), gc in zip(zip(path, path[1:]), gc_gaps):
            if e1 == e2 or e1[1] == e2[0]:
                inserted.append([])
                continue
            try:
                nodes = nx.shortest_path(self.Gp, e1[1], e2[0], weight="length")
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                inserted.append(None)
                ok = False
                continue
            seq = [min(((a, b, k) for k in self.Gp[a][b]), key=lambda e: self.length[e])
                   for a, b in zip(nodes, nodes[1:])]
            if sum(self.length[e] for e in seq) > self.max_factor * gc + self.slack:
                inserted.append(None)
                ok = False
                continue
            inserted.append(seq)
        return inserted, ok

    def route_points(self, lon, lat, times, path, inserted):
        """Weave observed fixes and inferred intermediate edges into one track.

        Observed fixes keep their own coordinates and timestamps; each inserted
        edge contributes one point at its start node, timed by how far along the
        inter-fix route it sits.  `observed` flags which is which.
        """
        out_lon, out_lat = [float(lon[0])], [float(lat[0])]
        out_t, out_e, obs = [float(times[0])], [path[0]], [True]

        for i, seq in enumerate(inserted):
            if seq:
                lens = np.array([self.length[e] for e in seq], float)
                cum = np.cumsum(np.r_[0.0, lens])
                total = cum[-1] if cum[-1] > 0 else 1.0
                t0, t1 = float(times[i]), float(times[i + 1])
                for j, e in enumerate(seq):
                    x, y = self.geom[e].coords[0][:2]
                    a, b = self._to_lonlat(x, y)
                    out_lon.append(float(a)); out_lat.append(float(b))
                    out_t.append(t0 + (cum[j] / total) * (t1 - t0))
                    out_e.append(e); obs.append(False)
            out_lon.append(float(lon[i + 1])); out_lat.append(float(lat[i + 1]))
            out_t.append(float(times[i + 1])); out_e.append(path[i + 1]); obs.append(True)

        return (np.asarray(out_lon), np.asarray(out_lat),
                np.maximum.accumulate(np.asarray(out_t, float)),
                out_e, np.asarray(obs))


def connectivity(seq):
    """Share of consecutive segment changes where the two segments share a node.

    After `fill_route` this should be 1.0 — it is a guard against a broken
    route, no longer a proxy for the sampling rate.
    """
    pairs = [(a, b) for a, b in zip(seq, seq[1:]) if a != b]
    if not pairs:
        return 1.0
    return sum(1 for (_, v1, _), (u2, _, _) in pairs if v1 == u2) / len(pairs)


# --- geometry -----------------------------------------------------------------
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dp, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dp / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))


def to_epoch(series):
    """Resolution-safe datetime -> unix seconds (pandas 2 and 3).

    NaT survives as NaN rather than raising: callers parse with
    `errors="coerce"` on purpose and drop the bad rows immediately after, so
    casting to int64 here would blow up before they get the chance.
    """
    tz = getattr(series.dtype, "tz", None)
    zero = pd.Timestamp("1970-01-01", tz=tz) if tz is not None else pd.Timestamp("1970-01-01")
    sec = (series - zero) // pd.Timedelta("1s")
    return sec.astype("int64") if sec.notna().all() else sec.astype("float64")

## 1. Load the weekly files

In [ ]:
USECOLS = ["DayNum", "VehId", "Trip", "Timestamp(ms)",
           "Latitude[deg]", "Longitude[deg]", "Vehicle Speed[km/h]",
           "Matchted Latitude[deg]", "Matched Longitude[deg]",   # sic: typo is in the data
           "Speed Limit[km/h]", "Elevation Smoothed[m]", "Gradient",
           "Energy_Consumption", "Intersection"]

files = sorted(glob.glob(os.path.join(RAW_DIR, "eVED_*_week.csv")))
print(len(files), "weekly files")

parts = []
for path in tqdm(files, desc="reading"):
    head = pd.read_csv(path, nrows=0)
    cols = [c for c in USECOLS if c in head.columns]
    parts.append(pd.read_csv(path, usecols=cols, low_memory=False))
raw = pd.concat(parts, ignore_index=True)
del parts
print(raw.shape)
raw.head(3)

In [ ]:
lat_col = "Matchted Latitude[deg]" if "Matchted Latitude[deg]" in raw.columns else "Matched Latitude[deg]"
lon_col = "Matched Longitude[deg]"

df = raw.rename(columns={lat_col: "lat", lon_col: "lon"})
df = df.dropna(subset=["lat", "lon", "DayNum", "Timestamp(ms)"])

# DayNum is the trip start (constant inside a trip); Timestamp(ms) is the clock inside it
zero = pd.Timestamp(EPOCH0).value // 10**9
df["epoch"] = (zero + (df["DayNum"] - 1) * 86400 + df["Timestamp(ms)"] / 1000).round().astype("int64")
df["trip_id"] = df["VehId"].astype("int64").astype(str) + "_" + df["Trip"].astype("int64").astype(str)
df["trip_key"] = df["trip_id"]

df = df[(df.lon.between(BBOX[0], BBOX[2])) & (df.lat.between(BBOX[1], BBOX[3]))]
df = df.sort_values(["trip_id", "epoch"]).drop_duplicates(["trip_id", "epoch"])
print(f"{len(df)} fixes, {df.trip_id.nunique()} trips, {df.VehId.nunique()} vehicles")
print(f"{pd.to_datetime(df.epoch.min(), unit='s')} -> {pd.to_datetime(df.epoch.max(), unit='s')}")

In [ ]:
# a sanity check worth keeping: how far the calibration moved the raw fixes
if "Latitude[deg]" in df.columns:
    off = haversine_m(df["Longitude[deg]"], df["Latitude[deg]"], df["lon"], df["lat"])
    print("raw -> matched offset (m):", off.quantile([.5, .9, .99]).round(1).to_dict())

gap = df.groupby("trip_id")["epoch"].diff()
print("sampling interval (s):", gap.quantile([.5, .9, .99]).round(2).to_dict())

## 2. Trip filters

In [ ]:
if RESAMPLE_S and RESAMPLE_S > 1:
    df = df[df.groupby("trip_id")["epoch"].transform(lambda s: s // RESAMPLE_S).diff().fillna(1) != 0]
    print("after resampling:", len(df), "fixes")

step = haversine_m(df["lon"].shift(), df["lat"].shift(), df["lon"], df["lat"])
step = step.where(df["trip_id"] == df["trip_id"].shift())
df["_dist"] = step

agg = df.groupby("trip_id").agg(n=("epoch", "size"), t0=("epoch", "first"),
                                t1=("epoch", "last"), meters=("_dist", "sum"))
agg["duration"] = agg.t1 - agg.t0
agg["mean_speed"] = agg.meters / agg.duration.replace(0, np.nan)

good = agg[(agg.n >= MIN_POINTS)
           & agg.duration.between(MIN_SECONDS, MAX_SECONDS)
           & (agg.meters >= MIN_METERS)
           & agg.mean_speed.between(*MEAN_SPEED_RANGE)].index
print(f"{len(good)} / {len(agg)} trips pass the filters")

if MAX_TRIPS and len(good) > MAX_TRIPS:
    good = pd.Series(good).sample(MAX_TRIPS, random_state=RANDOM_SEED).values
trips = df[df.trip_id.isin(set(good))]
agg.loc[good, ["n", "duration", "meters", "mean_speed"]].describe()

## 3. Road network

In [ ]:
# The Overpass download takes a few minutes and is cached as GraphML afterwards.
if os.path.exists(GRAPHML):
    G = ox.io.load_graphml(GRAPHML)
else:
    G = ox.graph.graph_from_bbox(BBOX, network_type="drive", simplify=True,
                                 truncate_by_edge=True)
    ox.io.save_graphml(G, GRAPHML)

G = ox.truncate.largest_component(G, strongly=True)
print(G)

## 4. Map matching

The coordinates are already on-road, so the matcher mostly has to pick *which*
edge and *which direction*. Snap distances should be small — if they are not,
the OSM network and the 2017 calibration have drifted apart and the bbox or the
`network_type` needs a look.

In [ ]:
# How far apart are the fixes *in space*?  Route ambiguity scales with distance,
# not time: on a 100 m grid, 150 m between fixes recovers ~81% of the true route
# whether that gap took 15 s at 10 m/s or 120 s at 1.2 m/s (measured).
#
# BUT: when the cadence is roughly fixed, distance = speed x cadence, so a gate
# on distance IS a gate on speed, and it throws away exactly the free-flow trips
# a travel-time model needs.  The table below therefore prints the speed limit
# each threshold implies at this dataset's own cadence.  Read it before setting
# MAX_MEDIAN_STEP_M to anything but None.
_step = haversine_m(trips["lon"].shift(), trips["lat"].shift(),
                    trips["lon"], trips["lat"])
_step = _step.where(trips["trip_key"] == trips["trip_key"].shift())
_dt = trips["epoch"].diff().where(trips["trip_key"] == trips["trip_key"].shift())
_med = _step.groupby(trips["trip_key"]).median().dropna()
_cad = float(_dt.median())

print(f"median cadence {_cad:.0f} s;  median distance between fixes, per trip (m):")
print(_med.describe(percentiles=[.05, .25, .5, .75, .95]).round(1).to_string())
print(f"\n{'MAX_MEDIAN_STEP_M':>18} {'route kept':>11} {'trips kept':>11} "
      f"{'implied speed cap':>18}")
for _thr, _rec in [(50, "100%"), (90, "99%"), (150, "81%"),
                   (200, "58%"), (300, "55%"), (600, "32%"), (900, "~25%")]:
    _cap = _thr / _cad * 3.6 if _cad > 0 else float("nan")
    print(f"{_thr:>15} m {_rec:>11} {(_med <= _thr).mean():>10.1%} "
          f"{_cap:>14.1f} km/h")
print("\nA low implied speed cap means the gate is selecting congested trips and\n"
      "biasing the label. Prefer None + the per-trip quality file, and filter\n"
      "downstream where you can see what it does to the travel-time distribution.")


In [ ]:
uid = assign_unique_osmids(G)
matcher = Matcher(G, radius_m=SNAP_RADIUS_M, sigma_m=SNAP_SIGMA_M,
                  beta_m=BETA_M, max_route_factor=MAX_ROUTE_FACTOR)

rows, observed, quality = [], [], []
rejected = {"sparse": 0, "no_candidate": 0, "snap": 0,
            "unroutable": 0, "short": 0, "connectivity": 0}

for trip_key, g in tqdm(list(trips.groupby("trip_key", sort=False)), desc="map matching"):
    lon, lat, ts = g["lon"].values, g["lat"].values, g["epoch"].values

    # Route ambiguity grows with the distance between fixes, so this is the
    # right axis — but at a fixed cadence it is also a speed filter, which
    # biases the very label we are predicting.  Off by default: the per-trip
    # median step is recorded instead, so the cut can be made downstream where
    # its effect on the travel-time distribution is visible.
    _s = haversine_m(lon[:-1], lat[:-1], lon[1:], lat[1:])
    _med_step = float(np.median(_s)) if len(_s) else 0.0
    if MAX_MEDIAN_STEP_M is not None and _med_step > MAX_MEDIAN_STEP_M:
        rejected["sparse"] += 1
        continue

    path, snap, keep = matcher.match(lon, lat)
    if path is None:
        rejected["no_candidate"] += 1
        continue
    lon, lat, ts = lon[keep], lat[keep], ts[keep]

    ok = snap <= MAX_SNAP_M
    if ok.mean() < MIN_MATCHED_FRAC:
        rejected["snap"] += 1
        continue
    lon, lat, ts = lon[ok], lat[ok], ts[ok]
    path = [e for e, m in zip(path, ok) if m]
    if len(path) < 2:
        rejected["short"] += 1
        continue

    p = gpd.GeoSeries(gpd.points_from_xy(lon, lat), crs="EPSG:4326").to_crs(matcher.crs)
    gc = np.hypot(np.diff(p.x.values), np.diff(p.y.values))
    inserted, routable = matcher.fill_route(path, gc)
    if not routable:
        rejected["unroutable"] += 1
        continue

    LON, LAT, T, E, obs = matcher.route_points(lon, lat, ts, path, inserted)
    if len(E) < MIN_POINTS or T[-1] - T[0] < MIN_SECONDS:
        rejected["short"] += 1
        continue
    if connectivity(E) < MIN_CONNECTIVITY:
        rejected["connectivity"] += 1
        continue

    stamps = np.maximum.accumulate(np.round(T).astype("int64")).tolist()
    observed.append(float(obs.mean()))
    quality.append({"Id": str(g["trip_id"].iloc[0]), "n_points": len(E),
                    "median_step_m": round(_med_step, 1),
                    "observed_share": round(float(obs.mean()), 4),
                    "duration_s": int(stamps[-1] - stamps[0])})
    rows.append({
        "Id": str(g["trip_id"].iloc[0]),
        "Coordinates": str([(round(float(a), 6), round(float(b), 6))
                            for a, b in zip(LON, LAT)]),
        "OSMids": str([uid[e]["unique_osmid"] for e in E]),
        "Timestamps": str(stamps),
        "Total_time": stamps[-1] - stamps[0],
    })

print(f"kept {len(rows)} trips; rejected {rejected}")
if observed:
    obs_mean = float(np.mean(observed))
    print(f"observed share of emitted points: {obs_mean:.1%} "
          f"(the rest are inferred route between fixes)")
    if obs_mean < 0.5:
        print("WARNING: most points are inferred, not observed. The sampling is too\n"
              "         coarse for a trustworthy route — treat this as OD data.")
if rejected["sparse"] > 0.5 * (len(rows) + sum(rejected.values())):
    print("\nMost trips were rejected as 'sparse'. That is a property of this\n"
          "dataset, not a bug — but note the gate is also a speed filter, so the\n"
          "survivors are the slow trips. Prefer MAX_MEDIAN_STEP_M = None and cut\n"
          "downstream, or switch to OD mode.")


## 5. Write the gold files

In [ ]:
matched = pd.DataFrame(rows, columns=["Id", "Coordinates", "OSMids", "Timestamps", "Total_time"])
matched.to_csv(f"{OUT}/matched_trips_{CITY}.csv", index=True)

edge_list = build_edge_list_directed(G, uid)
edge_list.to_csv(f"{OUT}/edge_list_directed_{CITY}.csv", index=False)

n_feat = build_road_network_geojson(G, uid, f"{OUT}/road_network_unique_osmids_{CITY}.geojson")
print(len(matched), "trips |", len(edge_list), "transitions |", n_feat, "segments")
matched.head(2)

# Per-trip matching quality, so the sparse/fast trips can be cut downstream
# with their effect on the travel-time distribution in view.
pd.DataFrame(quality).to_csv(f"{OUT}/trip_quality_{CITY}.csv", index=False)
print("quality file:", len(quality), "rows —",
      "median observed share",
      f"{pd.DataFrame(quality).observed_share.median():.1%}" if quality else "n/a")


In [ ]:
# eVED's real selling point is the extra channels — keep them alongside the trips
extra = [c for c in ["Vehicle Speed[km/h]", "Speed Limit[km/h]", "Elevation Smoothed[m]",
                     "Gradient", "Energy_Consumption", "Intersection"] if c in trips.columns]
if extra:
    num = trips[extra].apply(pd.to_numeric, errors="coerce")   # some columns carry "nan" as text
    num["trip_id"] = trips["trip_id"].values
    per_trip = num.groupby("trip_id").agg(["mean", "max"])
    per_trip.columns = ["_".join(c).strip() for c in per_trip.columns]
    per_trip.to_csv(f"{OUT}/trip_features_{CITY}.csv")
    print(per_trip.shape)
    per_trip.head(3)

In [ ]:
def validate_gold(trips_path, edge_list_path=None, geojson_path=None, require_coords=True):
    """Check the produced files against the Harbin/Omsk contract."""
    df = pd.read_csv(trips_path)
    problems = []

    expected = ["Unnamed: 0", "Id", "Coordinates", "OSMids", "Timestamps", "Total_time"]
    if list(df.columns) != expected:
        problems.append(f"columns are {list(df.columns)}, expected {expected}")

    bad_len = bad_eval = bad_total = bad_coord = 0
    osmids_seen = set()
    for _, r in df.iterrows():
        try:
            c = ast.literal_eval(r["Coordinates"])
            o = ast.literal_eval(r["OSMids"])
            t = ast.literal_eval(r["Timestamps"])
        except Exception:
            bad_eval += 1
            continue
        if not (len(c) == len(o) == len(t)):
            bad_len += 1
        if t[-1] - t[0] != r["Total_time"]:
            bad_total += 1
        if require_coords and not all(isinstance(p, tuple) and len(p) == 2 for p in c):
            bad_coord += 1
        osmids_seen.update(map(str, o))

    for label, n in [("rows that do not literal_eval", bad_eval),
                     ("rows with unequal list lengths", bad_len),
                     ("rows where Total_time != Timestamps[-1] - Timestamps[0]", bad_total),
                     ("rows with malformed coordinates", bad_coord)]:
        if n:
            problems.append(f"{n} {label}")

    print(f"{trips_path}: {len(df)} trips, {len(osmids_seen)} distinct segments, "
          f"Total_time median {df['Total_time'].median():.0f} s")

    if edge_list_path:
        el = pd.read_csv(edge_list_path, dtype=str)
        if list(el.columns) != ["osmid_u", "osmid_v"]:
            problems.append(f"edge list columns are {list(el.columns)}")
        known = set(el["osmid_u"]) | set(el["osmid_v"])
        missing = osmids_seen - known
        print(f"{edge_list_path}: {len(el)} transitions, "
              f"{len(osmids_seen & known)}/{len(osmids_seen)} trip segments present")
        if missing and len(missing) > 0.05 * max(len(osmids_seen), 1):
            problems.append(f"{len(missing)} trip segments missing from the edge list")

    if geojson_path:
        with open(geojson_path) as f:
            gj = json.load(f)
        keys = {f["properties"].get("unique_osmid", f["properties"]["osmid"])
                for f in gj["features"]}
        print(f"{geojson_path}: {len(gj['features'])} features, {len(keys)} unique ids")
        if not osmids_seen <= keys:
            problems.append(f"{len(osmids_seen - keys)} trip segments missing from the geojson")

    print("\nOK — matches the gold contract" if not problems
          else "\nPROBLEMS:\n  " + "\n  ".join(problems))
    return df

In [ ]:
_ = validate_gold(f"{OUT}/matched_trips_{CITY}.csv",
                  f"{OUT}/edge_list_directed_{CITY}.csv",
                  f"{OUT}/road_network_unique_osmids_{CITY}.geojson")

## Caveats

* **One mid-sized American city, 383 cars, 12 months.** Mostly the same people
  driving the same commutes. Great signal density, almost no diversity — this is a
  personalisation / energy dataset that happens to be excellent for R1, not a new
  "city" in the sense the table means.
* **Licence unstated.** VED is Apache-2.0; eVED says nothing. Ship the notebook,
  not the derived files, until that is settled.
* The `Matchted Latitude[deg]` spelling is a typo **in the data**; the notebook
  accepts both spellings.
* Trip ids are only unique per vehicle — the notebook builds `VehId_Trip`.
* The energy channels (`Energy_Consumption`, `Fuel Rate`, HV battery fields) are
  what makes joint time-and-energy prediction possible here and nowhere else. If
  that is not the plan, eVED is "just" a very clean 1 Hz R1 city.